# Tâche 4 Premier run de collecte (prototype)

**Consigne** : premier run de collecte sur un sous-ensemble de langues pour valider l'approche. Checkpoint S2 : revue du prototype et affinement du périmètre.

**Langues du prototype** : wolof (`wo`), yoruba (`yo`), n'ko (`nqo`), haoussa (`ha`), amharique (`am`).

**Ce que fait ce notebook**
1. Lit, pour chaque wiki, les informations utiles à l'extraction : noms locaux des espaces « Fichier » et « Catégorie », titre de la page d'accueil, préfixes des liens interlangues.
2. Télécharge les dumps `pages-articles` du **1er octobre 2026**, un à la fois, et vérifie leur empreinte SHA-1.
3. Lit chaque dump et garde toutes les pages de l'espace principal, hors redirection (page d'accueil comprise).
4. Extrait le texte avec les règles de la tâche 2 (`src/extraire_texte.py`). Les restes de balisage d'un wikitexte mal formé sont retirés et le texte est gardé.
5. Si le wikitexte ne donne aucun texte (page faite uniquement de modèles), récupère la page affichée avec l'API et en extrait le texte avec les mêmes règles (`src/extraire_html.py`).
6. Écrit le corpus (texte seul), le registre de provenance et le journal des pages écartées. Seuls les doublons exacts sont écartés : le premier texte est gardé, les copies sont écartées.
7. Valide le corpus (`src/valider_records.py`) et vérifie que chaque page du dump est comptée : soit dans le corpus, soit écartée avec une raison.
8. Calcule les mesures de référence par langue et les compare à l'inventaire de la tâche 1.



In [1]:
%pip install -q beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\akota\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import bz2
import hashlib
import json
import random
import re
import statistics
import sys
import time
import platform
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
import yaml


LANGUES = ["wo", "yo", "nqo", "ha", "am"]
CONTACT = "jerome.akota@afriklang.com"
USER_AGENT = f"projet_stage_nlp-bot/0.3 ({CONTACT}) python-requests/{requests.__version__}"
PAUSE_S = 0.2
ECHANTILLON_REVUE = 20          
GRAINE = 42                     

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from extraire_texte import extraire_texte          
from extraire_html import extraire_texte_html      
import valider_records as V                         
CONFIG = yaml.safe_load(open(ROOT / "config" / "config.yaml", encoding="utf-8"))
DUMP_DATE = CONFIG["dump_date"]
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
DUMPS_DIR = ROOT / CONFIG["paths"]["raw"] / "dumps" / DUMP_DATE
OUT_DIR = ROOT / CONFIG["paths"]["processed"] / "prototype" / RUN_ID
CORPUS_DIR = OUT_DIR / "corpus"
for d in (DUMPS_DIR, CORPUS_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert set(LANGUES) <= V.codes_autorises(), "Langue hors périmètre"
print("RUN_ID     :", RUN_ID)
print("Dumps      :", DUMPS_DIR)
print("Résultats  :", OUT_DIR)

RUN_ID     : 20261028T083452Z
Dumps      : C:\Users\akota\desktop\projet_stage_nlp\data\raw\dumps\20261001
Résultats  : C:\Users\akota\desktop\projet_stage_nlp\data\processed\prototype\20261028T083452Z


In [3]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})
JOURNAL = []   # journal des requêtes (provenance du run)


def _now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def get_json(url, params=None, max_retries=8):
    """GET JSON en série, avec maxlag et nouvelles tentatives (429, 5xx, coupure ou délai réseau dépassé)."""
    params = dict(params or {})
    if url.endswith("api.php"):
        params.update({"format": "json", "formatversion": "2", "maxlag": "5"})
    for attempt in range(max_retries):
        attente = min(2 ** attempt, 60)
        try:
            r = SESSION.get(url, params=params, timeout=(15, 90))
            JOURNAL.append({"url": r.url, "statut_http": r.status_code, "horodatage_utc": _now()})
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(int(r.headers.get("Retry-After", "0") or 0) or attente)
                continue
            r.raise_for_status()
            data = r.json()
        except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError, ValueError) as e:
            JOURNAL.append({"url": url, "statut_http": None, "erreur": type(e).__name__, "horodatage_utc": _now()})
            print(f"  réseau : {type(e).__name__}, nouvelle tentative dans {attente} s")
            time.sleep(attente)
            continue
        if isinstance(data, dict) and isinstance(data.get("error"), dict) and data["error"].get("code") == "maxlag":
            time.sleep(attente)
            continue
        time.sleep(PAUSE_S)
        return data
    raise RuntimeError(f"Échec après {max_retries} tentatives : {url}")

## Étape 1 Informations sur chaque wiki

Lues avec l'API `siteinfo` :
- les noms de l'espace « Fichier » (6) et de l'espace « Catégorie » (14) : nom local, nom canonique et alias ;
- le titre de la page d'accueil (gardée dans le corpus, comme les autres pages) ;
- les préfixes des liens interlangues (`[[fr:…]]`), retirés du texte.

In [4]:
INFOS = {}
for code in LANGUES:
    d = get_json(f"https://{code}.wikipedia.org/w/api.php",
                 {"action": "query", "meta": "siteinfo", "siprop": "general|namespaces|namespacealiases|interwikimap"})["query"]
    ns, alias = d["namespaces"], d.get("namespacealiases", [])
    noms = lambda i: sorted({n for n in [ns[str(i)].get("name"), ns[str(i)].get("canonical")] + [a["alias"] for a in alias if a["id"] == i] if n})
    INFOS[code] = {
        "dbname": d["general"]["wikiid"],
        "page_accueil": d["general"]["mainpage"],
        "noms_fichier": noms(6),
        "noms_categorie": noms(14),
        "interlangues": sorted(i["prefix"] for i in d.get("interwikimap", []) if i.get("language")),
    }
pd.DataFrame({c: {"dbname": v["dbname"], "page d'accueil": v["page_accueil"],
                  "Fichier": ", ".join(v["noms_fichier"]), "Catégorie": ", ".join(v["noms_categorie"]),
                  "préfixes interlangues": len(v["interlangues"])} for c, v in INFOS.items()}).T

,dbname,page d'accueil,Fichier,Catégorie,préfixes interlangues
wo,wowiki,Xët wu njëkk,"Dencukaay, File, Image","Category, Catégorie, Wàll",378
yo,yowiki,Ojúewé Àkọ́kọ́,"File, Fáìlì, Image, Àwòrán","Category, Ẹ̀ka",378
nqo,nqowiki,ߓߏ߬ߟߏ߲߬ߘߊ,"File, Image, ߞߐߕߐ߮","Category, ߦߌߟߡߊ",378
ha,hawiki,Babban shafi,"Fayil, File, Image","Category, Rukuni",378
am,amwiki,ዋናው ገጽ,"File, Image, ስዕል","Category, መደብ",378


## Étape 2 Téléchargement des dumps (1er octobre 2026)

Pour chaque wiki  lecture de `dumpstatus.json`, puis téléchargement des fichiers `pages-articles` **un à la fois** vérification de leur empreinte SHA-1. Un fichier déjà présent et correct n'est pas retéléchargé.

In [5]:
def sha1_fichier(chemin):
    h = hashlib.sha1()
    with open(chemin, "rb") as f:
        for bloc in iter(lambda: f.read(1 << 20), b""):
            h.update(bloc)
    return h.hexdigest()


def telecharger_dump(code):
    db = INFOS[code]["dbname"]
    st = get_json(f"https://dumps.wikimedia.org/{db}/{DUMP_DATE}/dumpstatus.json")
    job = st["jobs"]["articlesdump"]
    assert job["status"] == "done", f"{db} : dump {DUMP_DATE} non terminé ({job['status']})"
    fichiers = []
    for nom, meta in sorted(job["files"].items()):
        chemin = DUMPS_DIR / nom
        if not (chemin.exists() and sha1_fichier(chemin) == meta["sha1"]):
            for essai in range(5):                          
                try:
                    with SESSION.get("https://dumps.wikimedia.org" + meta["url"], stream=True, timeout=(15, 120)) as r:
                        JOURNAL.append({"url": r.url, "statut_http": r.status_code, "horodatage_utc": _now()})
                        r.raise_for_status()
                        with open(chemin, "wb") as f:
                            for bloc in r.iter_content(1 << 20):
                                f.write(bloc)
                    break
                except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError) as e:
                    print(f"  {nom} : {type(e).__name__}, nouvelle tentative dans {30 * (essai + 1)} s")
                    time.sleep(30 * (essai + 1))
            else:
                raise RuntimeError(f"Téléchargement impossible : {nom}")
            assert sha1_fichier(chemin) == meta["sha1"], f"SHA-1 incorrect pour {nom}"
        fichiers.append({"code": code, "fichier": nom, "taille_mo": round(meta["size"] / 1e6, 1), "sha1": meta["sha1"]})
    return fichiers

DUMPS = [f for code in LANGUES for f in telecharger_dump(code)]
pd.DataFrame(DUMPS)

,code,fichier,taille_mo,sha1
0,wo,wowiki-20261001-pages-articles.xml.bz2,2.3,53cf4066774fe8107f4c5e792dd864a4800759f6
1,yo,yowiki-20261001-pages-articles.xml.bz2,29.5,3d802e68f117370c1024a8b1f134f0c91f37db49
2,nqo,nqowiki-20261001-pages-articles.xml.bz2,2.1,81aa130e42dceb1c37f72035003afc99e0876f1c
3,ha,hawiki-20261001-pages-articles.xml.bz2,186.2,ea0b546ee881133b5da0c60e31cdd62b75163e82
4,am,amwiki-20261001-pages-articles.xml.bz2,9.2,c50221525e3347ff70787ecdf22c6f30951b786d


## Étape 3 Lecture des dumps et extraction du texte

Pour chaque page de l'espace principal (namespace 0), hors redirection, page d'accueil comprise :
1. le texte est extrait du wikitexte ; `{{PAGENAME}}` est remplacé par le titre de la page et les restes de balisage d'un wikitexte mal formé sont retirés ;
2. si le wikitexte ne donne aucun texte (page faite uniquement de modèles), la page affichée est demandée à l'API (`action=parse` sur la même révision) et son texte est extrait avec les mêmes règles ;
3. un texte identique à un texte déjà gardé est écarté (doublon exact) : seul le premier est gardé.

Une page n'est écartée que dans deux cas : doublon exact, ou aucun texte même dans la page affichée. Le corpus ne contient que le texte ; la provenance et les pages écartées vont dans des fichiers séparés.

In [ ]:
def pages_du_dump(chemin):
    """Parcourt un dump XML compressé (bz2) page par page, sans le charger en mémoire."""
    with bz2.open(chemin, "rb") as f:
        for _, el in ET.iterparse(f, events=("end",)):
            if el.tag.rsplit("}", 1)[-1] != "page":
                continue
            champ = lambda nom: next((c for c in el.iter() if c.tag.rsplit("}", 1)[-1] == nom), None)
            rev = champ("revision")
            texte = champ("text")
            modele = champ("model")
            yield {
                "titre": champ("title").text,
                "ns": int(champ("ns").text),
                "page_id": int(champ("id").text),
                "redirection": champ("redirect") is not None,
                "revision_id": int(next(c for c in rev if c.tag.rsplit("}", 1)[-1] == "id").text),
                "revision_date": champ("timestamp").text,
                "modele": modele.text if modele is not None else "wikitext",
                "wikitexte": (texte.text or "") if texte is not None else "",
            }
            el.clear()


DATE_API = datetime.now(timezone.utc).strftime("%Y%m%d")


def texte_page_affichee(code, revision_id):
    """Texte de la page affichée (API action=parse) ; (None, raison) si l'API ne la rend pas."""
    d = get_json(f"https://{code}.wikipedia.org/w/api.php",
                 {"action": "parse", "oldid": revision_id, "prop": "text",
                  "disablelimitreport": "1", "disableeditsection": "1"})
    if "error" in d:
        return None, f"page affichée non disponible ({d['error'].get('code')})"
    return extraire_texte_html(d["parse"]["text"]), None


def traiter_langue(code):
    infos = INFOS[code]
    source_dump = f"{infos['dbname']}-{DUMP_DATE}-pages-articles"
    vus, corpus, provenance, exclusions = set(), [], [], []
    compte = Counter()
    fichiers = [DUMPS_DIR / f["fichier"] for f in DUMPS if f["code"] == code]
    for chemin in fichiers:
        for p in pages_du_dump(chemin):
            compte["pages_dump"] += 1
            if p["ns"] != 0 or p["redirection"]:
                continue                                   # hors périmètre  non compté dans le corpus
            compte["pages_ns0"] += 1
            url = f"https://{code}.wikipedia.org/w/index.php?oldid={p['revision_id']}"
            raison, source = None, source_dump
            t = (extraire_texte(p["wikitexte"], infos["noms_fichier"], infos["noms_categorie"], infos["interlangues"], titre=p["titre"])
                 if p["modele"] == "wikitext" else "")
            if not t:                                      # page faite de modèles texte de la page affichée
                t, raison = texte_page_affichee(code, p["revision_id"])
                source = f"api-{DATE_API}"
                compte["textes_api"] += 1
                if raison is None and not t:
                    raison = "aucun texte, même dans la page affichée"
            if raison is None:
                erreurs = V.regles_texte(t)
                if erreurs:                                # ne doit pas arriver le nettoyage retire ces restes
                    raison = "règle enfreinte : " + erreurs[0]
                else:
                    h = V.sha256(t)
                    if h in vus:
                        raison = "doublon exact d'un texte déjà gardé"
                    else:
                        vus.add(h)
                        corpus.append({"text": t})
                        provenance.append({"text_sha256": h, "language_code": code, "source_url": url,
                                           "source_dump": source, "revision_date": p["revision_date"],
                                           "collected_at": _now(), "license": "CC-BY-SA-4.0", "quality_flags": []})
            if raison:
                exclusions.append({"langue": code, "titre": p["titre"], "source_url": url, "raison": raison,
                                   "extrait": (p["wikitexte"] or "")[:300]})
    return corpus, provenance, exclusions, compte


RESULTATS = {}
for code in LANGUES:
    t0 = time.time()
    corpus, provenance, exclusions, compte = traiter_langue(code)
    with open(CORPUS_DIR / f"{code}.jsonl", "w", encoding="utf-8") as f:
        for ligne in corpus:
            f.write(json.dumps(ligne, ensure_ascii=False) + "\n")
    RESULTATS[code] = {"provenance": provenance, "exclusions": exclusions, "compte": compte,
                       "mots": [len(l["text"].split()) for l in corpus], "duree_s": round(time.time() - t0)}
    print(f"{code} : {compte['pages_ns0']} pages, {len(corpus)} textes, {len(exclusions)} écartées, "
          f"{compte['textes_api']} pages lues par l'API ({RESULTATS[code]['duree_s']} s)")

with open(OUT_DIR / "provenance.jsonl", "w", encoding="utf-8") as f:
    for code in LANGUES:
        for p in RESULTATS[code]["provenance"]:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
with open(OUT_DIR / "exclusions.jsonl", "w", encoding="utf-8") as f:
    for code in LANGUES:
        for e in RESULTATS[code]["exclusions"]:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

wo : 1812 pages, 1809 textes, 3 écartées, 7 pages lues par l'API (16 s)


## Étape 4 Validation et comptabilité des pages

- **Validation** du corpus et du registre avec les critères de la tâche 2 (A1 à A5).
- **Comptabilité (critère C2)** : pour chaque langue, chaque page de l'espace principal hors redirection doit être soit dans le corpus, soit dans le journal d'exclusions.

In [ ]:
resultat, erreurs = V.valider_corpus(CORPUS_DIR, OUT_DIR / "provenance.jsonl")
print(f"Validation : {resultat['lignes_valides']}/{resultat['lignes']} lignes valides, "
      f"{resultat['avec_provenance']}/{resultat['lignes']} avec provenance")
for e in erreurs[:20]:
    print(" -", e)

comptabilite = {code: {"pages_ns0": r["compte"]["pages_ns0"], "textes": len(r["mots"]),
                       "exclusions": len(r["exclusions"]),
                       "toutes_comptees": r["compte"]["pages_ns0"] == len(r["mots"]) + len(r["exclusions"])}
                for code, r in RESULTATS.items()}
CONTROLES = {
    "corpus valide (A1-A5)": not erreurs,
    "chaque page comptée (C2)": all(v["toutes_comptees"] for v in comptabilite.values()),
    "chaque langue a des textes (C1)": all(v["textes"] > 0 for v in comptabilite.values()),
}
for nom, ok in CONTROLES.items():
    print("OK   " if ok else "ÉCHEC", nom)
pd.DataFrame(comptabilite).T

## Étape 5 Mesures de référence (baseline) et comparaison avec la tâche 1

Par langue : nombre de textes, exclusions par raison, nombre total de mots (découpage sur les espaces), médiane et moyenne de mots par texte. Comparaison avec l'inventaire de la tâche 1 : nombre officiel d'articles et nombre de mots compté par le moteur de recherche de Wikipédia.

In [ ]:
inv_dirs = sorted((ROOT / CONFIG["paths"]["processed"] / "inventaire").glob("2*"))
inventaire = {}
if inv_dirs and (inv_dirs[-1] / "inventaire_synthese.json").exists():
    for r in json.load(open(inv_dirs[-1] / "inventaire_synthese.json", encoding="utf-8")):
        inventaire[r["code"]] = r
    print("Inventaire de référence :", inv_dirs[-1].name)

mesures = []
for code, r in RESULTATS.items():
    m = r["mots"]
    inv = inventaire.get(code, {})
    mesures.append({
        "langue": code,
        "pages_ns0": r["compte"]["pages_ns0"],
        "textes": len(m),
        "exclusions": len(r["exclusions"]),
        "pages_lues_api": r["compte"]["textes_api"],
        "textes_api": sum(p["source_dump"].startswith("api-") for p in r["provenance"]),
        "mots_total": sum(m),
        "mots_mediane": statistics.median(m) if m else 0,
        "mots_moyenne": round(statistics.mean(m), 1) if m else 0,
        "articles_tache1": inv.get("articles"),
        "mots_tache1": inv.get("mots_articles"),
        "taille_dump_mo": sum(f["taille_mo"] for f in DUMPS if f["code"] == code),
        "duree_s": r["duree_s"],
    })
mesures = pd.DataFrame(mesures).set_index("langue")
mesures["mots_vs_tache1_pct"] = (100 * mesures["mots_total"] / mesures["mots_tache1"]).round(1)
mesures

In [ ]:
raisons = pd.DataFrame([{"langue": code, "raison": e["raison"]} for code, r in RESULTATS.items() for e in r["exclusions"]])
raisons_par_langue = (raisons.value_counts().unstack(0).fillna(0).astype(int)
                      if len(raisons) else pd.DataFrame())
raisons_par_langue

## Étape 6 Échantillon pour la relecture (checkpoint S2)

`ECHANTILLON_REVUE` textes tirés au hasard par langue (graine fixe), avec leur source, pour vérifier à la main la qualité de l'extraction ; plus jusqu'à 10 textes venus de la page affichée (API) et 10 pages écartées par langue.

In [ ]:
rng = random.Random(GRAINE)
revue = []
for code in LANGUES:
    lignes = [json.loads(l) for l in open(CORPUS_DIR / f"{code}.jsonl", encoding="utf-8")]
    prov = RESULTATS[code]["provenance"]
    for i in sorted(rng.sample(range(len(lignes)), min(ECHANTILLON_REVUE, len(lignes)))):
        revue.append({"type": "texte gardé", "langue": code, "source_url": prov[i]["source_url"], "texte": lignes[i]["text"]})
    api = [i for i, p in enumerate(prov) if p["source_dump"].startswith("api-")]
    for i in sorted(rng.sample(api, min(10, len(api)))):
        revue.append({"type": "texte gardé (page affichée)", "langue": code, "source_url": prov[i]["source_url"], "texte": lignes[i]["text"]})
    excl = RESULTATS[code]["exclusions"]
    for e in rng.sample(excl, min(10, len(excl))):
        revue.append({"type": "exclusion", "langue": code, "source_url": e["source_url"], "raison": e["raison"], "extrait_wikitexte": e["extrait"]})
with open(OUT_DIR / "echantillon_revue.jsonl", "w", encoding="utf-8") as f:
    for r in revue:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(len(revue), "éléments dans l'échantillon de relecture")
for r in [x for x in revue if x["type"] == "texte gardé"][:3]:
    print("\n---", r["langue"], r["source_url"])
    print(r["texte"][:500])

## Étape 7 Sauvegarde du run

Fichiers produits dans `data/processed/prototype/<RUN_ID>/` :
- `corpus/<langue>.jsonl` : le corpus, texte seul ;
- `provenance.jsonl` : le registre de provenance (hors corpus) ;
- `exclusions.jsonl` : les pages écartées (doublons exacts, pages sans aucun texte) et leur raison ;
- `mesures.json`, `raisons_exclusion.json` : les mesures de référence ;
- `echantillon_revue.jsonl` : l'échantillon de relecture ;
- `run_metadata.json` : date, versions, paramètres, dumps utilisés, contrôles ;
- `journal_requetes.jsonl` : les requêtes du run.

In [ ]:
mesures.reset_index().to_json(OUT_DIR / "mesures.json", orient="records", force_ascii=False, indent=2)
(raisons_par_langue.reset_index().to_json(OUT_DIR / "raisons_exclusion.json", orient="records", force_ascii=False, indent=2)
 if len(raisons_par_langue) else None)
with open(OUT_DIR / "journal_requetes.jsonl", "w", encoding="utf-8") as f:
    for j in JOURNAL:
        f.write(json.dumps(j, ensure_ascii=False) + "\n")
import mwparserfromhell, bs4
meta = {
    "run_id": RUN_ID, "fin_utc": _now(), "langues": LANGUES, "dump_date": DUMP_DATE, "dumps": DUMPS,
    "python": sys.version.split()[0], "plateforme": platform.platform(), "requests": requests.__version__,
    "mwparserfromhell": mwparserfromhell.__version__, "pandas": pd.__version__, "beautifulsoup4": bs4.__version__,
    "parametres": {"ECHANTILLON_REVUE": ECHANTILLON_REVUE, "GRAINE": GRAINE, "PAUSE_S": PAUSE_S},
    "controles": CONTROLES, "validation": resultat, "erreurs_validation": erreurs[:50],
    "infos_wikis": {c: {k: v for k, v in i.items() if k != "interlangues"} for c, i in INFOS.items()},
}
with open(OUT_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Run sauvegardé dans", OUT_DIR)